### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Unsloth

In [3]:
import json
import random
from pathlib import Path

# ── Konfigurasi ───────────────────────────────────────────────────────────

DATASET_DIR = Path("/content/drive/MyDrive/dataset")
OUTPUT_DIR = Path("dataset_prepared")
VAL_SPLIT = 0.15
SEED = 3407

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# Nama folder -> (label singkat, deskripsi yang akan dipelajari model)
# Edit / tambah baris di sini sesuai jenis error yang kamu punya.
LABELS = {
    "normal":   {
        "class": "normal",
        "text": "Kondisi normal. LED module menyala dan berfungsi dengan baik, tidak ada error.",
    },
    "blocking": {
        "class": "blocking",
        "text": "Error: blocking. LED mengalami Kegagalan fungsional hardware (Error: Blocking) terdeteksi pada modul layar LED, ditandai dengan adanya satu atau beberapa modul panel yang mati, mengalami distorsi sinyal, atau membeku (freeze) sehingga membentuk blok baris hitam statis yang merusak integritas frame konten yang sedang berjalan."
    },
    "dead": {
        "class": "dead",
        "text": "Error: dead/mati. Sebagian atau seluruh segmen/pixel LED module tidak menyala sama sekali.",
    },
}

PROMPT = (
    "Analisis gambar LED module berikut. Tentukan apakah kondisinya normal atau "
    "terdapat error (blocking / dead / lainnya), lalu jelaskan singkat."
)

# ── Proses ────────────────────────────────────────────────────────────────

def collect_samples():
    samples = []
    for folder_name, meta in LABELS.items():
        folder = DATASET_DIR / folder_name
        if not folder.exists():
            print(f"[skip] folder '{folder}' tidak ditemukan, dilewati.")
            continue

        files = [
            p for p in folder.rglob("*")
            if p.suffix.lower() in IMAGE_EXTS
        ]
        print(f"[{folder_name}] ditemukan {len(files)} gambar")

        for f in files:
            samples.append({
                "image_path": str(f.resolve()),
                "class": meta["class"],
                "label_text": meta["text"],
            })
    return samples


def main():
    random.seed(SEED)
    samples = collect_samples()

    if not samples:
        raise SystemExit(
            "Tidak ada gambar ditemukan. Pastikan folder dataset/normal, "
            "dataset/blocking, dataset/dead sudah berisi gambar."
        )

    random.shuffle(samples)

    # Cek keseimbangan kelas
    from collections import Counter
    counts = Counter(s["class"] for s in samples)
    print("\nDistribusi kelas:")
    for cls, n in counts.items():
        print(f"  {cls}: {n}")
    min_count = min(counts.values())
    max_count = max(counts.values())
    if max_count > min_count * 3:
        print(
            "\n[!] PERINGATAN: dataset kamu tidak seimbang "
            f"(kelas terbanyak {max_count} vs tersedikit {min_count}). "
            "Model bisa bias ke kelas mayoritas. Idealnya jumlah tiap kelas mirip, "
            "atau lakukan oversampling pada kelas minoritas.\n"
        )

    n_val = max(1, int(len(samples) * VAL_SPLIT))
    val_samples = samples[:n_val]
    train_samples = samples[n_val:]

    OUTPUT_DIR.mkdir(exist_ok=True)

    def write_jsonl(path, data):
        with open(path, "w", encoding="utf-8") as f:
            for s in data:
                record = {
                    "image_path": s["image_path"],
                    "prompt": PROMPT,
                    "response": s["label_text"],
                    "class": s["class"],
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    write_jsonl(OUTPUT_DIR / "train.jsonl", train_samples)
    write_jsonl(OUTPUT_DIR / "val.jsonl", val_samples)

    print(f"\nSelesai. {len(train_samples)} data train, {len(val_samples)} data val.")
    print(f"Disimpan di: {OUTPUT_DIR}/train.jsonl dan {OUTPUT_DIR}/val.jsonl")


if __name__ == "__main__":
    main()

[normal] ditemukan 43 gambar
[blocking] ditemukan 64 gambar
[dead] ditemukan 18 gambar

Distribusi kelas:
  blocking: 64
  dead: 18
  normal: 43

[!] PERINGATAN: dataset kamu tidak seimbang (kelas terbanyak 64 vs tersedikit 18). Model bisa bias ke kelas mayoritas. Idealnya jumlah tiap kelas mirip, atau lakukan oversampling pada kelas minoritas.


Selesai. 107 data train, 18 data val.
Disimpan di: dataset_prepared/train.jsonl dan dataset_prepared/val.jsonl


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
import json

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from datasets import Dataset
from PIL import Image
from trl import SFTTrainer, SFTConfig

# ── Konfigurasi ───────────────────────────────────────────────────────────

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_LENGTH = 2048
OUTPUT_ADAPTER_DIR = "led_defect_lora"

# load_in_4bit=False -> 16bit LoRA (default resmi Unsloth untuk model ini).
# Kalau nanti kena OOM di GPU kecil, baru ganti ke True (QLoRA/4-bit).
LOAD_IN_4BIT = False

# LoRA — vision layer WAJIB nyala untuk task deteksi visual seperti ini
LORA_CONFIG = dict(
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

TRAIN_CONFIG = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=7,       # naikkan kalau dataset kecil (<200 gambar), turunkan kalau besar
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="epoch",
    output_dir="train_outputs",
    report_to="none",
)

# ── Load & format dataset ───────────────────────────────────────────────

def load_jsonl_as_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def to_conversation(record):
    """Ubah 1 record jadi format chat multimodal yang dipahami Qwen3.5."""
    image = Image.open(record["image_path"]).convert("RGB")
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": record["prompt"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": record["response"]},
                ],
            },
        ]
    }


def main():
    train_records = load_jsonl_as_dataset("dataset_prepared/train.jsonl")
    val_records = load_jsonl_as_dataset("dataset_prepared/val.jsonl")

    print(f"Memuat {len(train_records)} data train, {len(val_records)} data val...")

    train_data = [to_conversation(r) for r in train_records]
    train_dataset = Dataset.from_list(train_data)

    # ── Load model ────────────────────────────────────────────────────
    print("Memuat model...")
    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_NAME,
        load_in_4bit=LOAD_IN_4BIT,
        use_gradient_checkpointing="unsloth",  # setting asli dari notebook resmi
    )

    model = FastVisionModel.get_peft_model(model, **LORA_CONFIG)

    FastVisionModel.for_training(model)  # WAJIB dipanggil sebelum training

    # ── Setup trainer ────────────────────────────────────────────────
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),  # WAJIB pakai ini
        train_dataset=train_dataset,
        args=SFTConfig(
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},  # WAJIB untuk vision finetuning
            max_length=MAX_LENGTH,
            **TRAIN_CONFIG,
        ),
    )

    # ── Train ────────────────────────────────────────────────────────
    print("Mulai training...")
    trainer.train()

    # ── Simpan LoRA adapter ─────────────────────────────────────────
    model.save_pretrained(OUTPUT_ADAPTER_DIR)
    tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)
    print(f"\nSelesai. LoRA adapter disimpan di: {OUTPUT_ADAPTER_DIR}/")
    print("Lanjut ke 3_evaluate.py untuk cek akurasi, lalu 4_export_gguf.py untuk export.")


if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Memuat 107 data train, 18 data val...
